# Differential Equations — Session 36
## Section 8.1: Theory of Linear Systems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. write a system in matrix form;
2. distinguish homogeneous and nonhomogeneous systems;
3. convert a higher-order equation to a first-order system;
4. verify a solution vector;
5. state existence and uniqueness for linear systems;
6. apply superposition;
7. test linear independence using a determinant;
8. define a fundamental set and fundamental matrix;
9. express the general solution of homogeneous and nonhomogeneous systems.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | First-order systems and matrix form |
| 15–28 min | Higher-order equation as a system |
| 28–43 min | Solution vectors and IVPs |
| 43–58 min | Existence, uniqueness, and superposition |
| 58–76 min | Linear independence and fundamental matrices |
| 76–88 min | Nonhomogeneous structure |
| 88–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad_vec
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def solve_linear_system(A, x0, t_span=(-5, 5), points=1200, forcing=None):
    A = np.asarray(A, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    t_eval = np.linspace(t_span[0], t_span[1], points)

    if forcing is None:
        def rhs(t, x):
            return A @ x
    else:
        def rhs(t, x):
            return A @ x + np.asarray(forcing(t), dtype=float)

    return solve_ivp(rhs, t_span, x0, t_eval=t_eval, rtol=1e-9, atol=1e-11)

def vector_field(A, xlim=(-4, 4), ylim=(-4, 4), density=21):
    A = np.asarray(A, dtype=float)
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    U = A[0,0]*X + A[0,1]*Y
    V = A[1,0]*X + A[1,1]*Y
    speed = np.sqrt(U**2 + V**2)
    U = np.divide(U, speed, out=np.zeros_like(U), where=speed>1e-12)
    V = np.divide(V, speed, out=np.zeros_like(V), where=speed>1e-12)
    plt.quiver(X, Y, U, V, speed)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x")
    plt.ylabel("y")

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 8.1-A — First-order system

A first-order system has the normal form

$$
\mathbf X'=\mathbf G(t,\mathbf X).
$$

A linear system has the form

$$
\mathbf X'=A(t)\mathbf X+\mathbf F(t).
$$

It is **homogeneous** when $\mathbf F(t)=\mathbf 0$.

### Definition 8.1-B — Solution vector

A differentiable vector function $\mathbf X(t)$ is a solution when it satisfies the system at every point in the interval.

### Theorem 8.1-C — Existence and uniqueness

If all entries of $A(t)$ and $\mathbf F(t)$ are continuous on an interval $I$ containing $t_0$, then

$$
\mathbf X'=A(t)\mathbf X+\mathbf F(t),
\qquad
\mathbf X(t_0)=\mathbf X_0
$$

has a unique solution on $I$.

### Theorem 8.1-D — Superposition

If $\mathbf X_1,\ldots,\mathbf X_k$ solve the homogeneous system, then

$$
c_1\mathbf X_1+\cdots+c_k\mathbf X_k
$$

also solves it.

### Definition 8.1-E — Linear independence

Solution vectors $\mathbf X_1,\ldots,\mathbf X_n$ are linearly independent when

$$
c_1\mathbf X_1+\cdots+c_n\mathbf X_n=\mathbf 0
$$

implies $c_1=\cdots=c_n=0$.

### Theorem 8.1-F — Determinant criterion

Let

$$
\Phi(t)=
\begin{pmatrix}
|& &|\\
\mathbf X_1(t)&\cdots&\mathbf X_n(t)\\
|& &|
\end{pmatrix}.
$$

The solutions are linearly independent if and only if

$$
\det\Phi(t)\ne0
$$

throughout the interval.

### Definition 8.1-G — Fundamental set and fundamental matrix

A set of $n$ independent solutions of an $n$-dimensional homogeneous system is a **fundamental set**. The matrix formed from these columns is a **fundamental matrix**.

### Theorem 8.1-H — General solution

For a homogeneous system,

$$
\mathbf X=\Phi(t)\mathbf C.
$$

For a nonhomogeneous system,

$$
\mathbf X=\mathbf X_c+\mathbf X_p.
$$

### Classroom Checkpoint — Fundamental Matrix Test

What condition on a solution matrix $\Phi(t)$ guarantees that its columns form a fundamental set?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Matrix form

The system

$$
\begin{aligned}
x'&=2x-3y+\sin t,\\
y'&=x+4y+t
\end{aligned}
$$

becomes

$$
\mathbf X'
=
\begin{pmatrix}
2&-3\\
1&4
\end{pmatrix}
\mathbf X
+
\begin{pmatrix}
\sin t\\
t
\end{pmatrix}.
$$

In [ ]:
A = np.array([[2, -3], [1, 4]], dtype=float)
print("Coefficient matrix A:")
print(A)

## 2. Converting a higher-order equation to a system

For

$$
y'''+2y''-y'+5y=e^{-t},
$$

let

$$
x_1=y,\qquad x_2=y',\qquad x_3=y''.
$$

Then

$$
\begin{aligned}
x_1'&=x_2,\\
x_2'&=x_3,\\
x_3'&=-5x_1+x_2-2x_3+e^{-t}.
\end{aligned}
$$

In [ ]:
def rhs_third_order(t, x):
    return [x[1], x[2], -5*x[0]+x[1]-2*x[2]+np.exp(-t)]

sol = solve_ivp(rhs_third_order, (0, 12), [1, 0, -1],
                t_eval=np.linspace(0, 12, 900), rtol=1e-9, atol=1e-11)

plt.plot(sol.t, sol.y[0], label=r"$y$")
plt.plot(sol.t, sol.y[1], label=r"$y'$")
plt.plot(sol.t, sol.y[2], label=r"$y''$")
plt.legend()
plt.title("One third-order equation as a first-order system")
plt.show()

## 3. Verifying a solution vector

For

$$
A=
\begin{pmatrix}
1&1\\
0&1
\end{pmatrix},
$$

consider

$$
\mathbf X_1(t)=e^t
\begin{pmatrix}
1\\0
\end{pmatrix},
\qquad
\mathbf X_2(t)=e^t
\begin{pmatrix}
t\\1
\end{pmatrix}.
$$

In [ ]:
t = sp.symbols("t", real=True)
A_sym = sp.Matrix([[1, 1], [0, 1]])
X1 = sp.exp(t)*sp.Matrix([1, 0])
X2 = sp.exp(t)*sp.Matrix([t, 1])

print("Residual for X1:")
display(sp.simplify(sp.diff(X1, t)-A_sym*X1))
print("Residual for X2:")
display(sp.simplify(sp.diff(X2, t)-A_sym*X2))

## 4. Superposition is visible geometrically

Every initial condition produces one trajectory. Changing the constants changes the linear combination of fundamental solutions.

In [ ]:
def superposition_explorer(c1=1.0, c2=0.5):
    t = np.linspace(-2, 2, 700)
    x = np.exp(t)*(c1+c2*t)
    y = c2*np.exp(t)

    plt.plot(x, y, linewidth=2)
    plt.scatter([x[0]], [y[0]], s=60, label="start")
    plt.xlabel("x(t)")
    plt.ylabel("y(t)")
    plt.title(fr"$c_1={c1:.2f},\;c_2={c2:.2f}$")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        superposition_explorer,
        c1=FloatSlider(min=-3, max=3, step=0.25, value=1),
        c2=FloatSlider(min=-3, max=3, step=0.25, value=0.5)
    )
else:
    superposition_explorer()

## 5. Fundamental matrix and determinant

For the two solutions above,

$$
\Phi(t)
=
e^t
\begin{pmatrix}
1&t\\
0&1
\end{pmatrix},
$$

so

$$
\det\Phi(t)=e^{2t}\ne0.
$$

In [ ]:
t_grid = np.linspace(-3, 3, 600)
wronskian = np.exp(2*t_grid)

plt.semilogy(t_grid, wronskian)
plt.xlabel("t")
plt.ylabel(r"$|\det\Phi(t)|$")
plt.title("A nonzero fundamental determinant")
plt.show()

## 6. Vector field and unique trajectories

Uniqueness prevents two distinct solution trajectories from crossing at the same point at the same time.

In [ ]:
A = np.array([[-1, 2], [-2, -1]], dtype=float)
vector_field(A, (-4, 4), (-4, 4))

for x0 in ([3,0], [2,2], [-3,1], [0,-3]):
    sol = solve_linear_system(A, x0, (0, 6), points=600)
    plt.plot(sol.y[0], sol.y[1])

plt.title("Representative trajectories of a homogeneous system")
plt.show()

## 7. Nonhomogeneous structure

For

$$
\mathbf X'=A\mathbf X+\mathbf b,
$$

a constant equilibrium particular solution satisfies

$$
A\mathbf X_p+\mathbf b=\mathbf 0.
$$

Every solution is

$$
\mathbf X(t)=\mathbf X_c(t)+\mathbf X_p.
$$

In [ ]:
def nonhomogeneous_explorer(force=1.0):
    A = np.array([[-2, 1], [0, -1]], dtype=float)
    b = np.array([force, 2*force], dtype=float)
    xp = -np.linalg.solve(A, b)

    def forcing(t):
        return b

    for x0 in ([3,0], [-2,3], [0,-2]):
        sol = solve_linear_system(A, x0, (0, 8), points=700, forcing=forcing)
        plt.plot(sol.y[0], sol.y[1])

    plt.scatter([xp[0]], [xp[1]], s=100, label="equilibrium particular solution")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("All trajectories are shifted toward the same equilibrium")
    plt.legend()
    plt.show()
    print("particular equilibrium:", xp)

if WIDGETS_AVAILABLE:
    interact(
        nonhomogeneous_explorer,
        force=FloatSlider(min=-3, max=3, step=0.25, value=1)
    )
else:
    nonhomogeneous_explorer()

## Classroom Checkpoint — Exit Check

Convert

$$
y''+4y'+3y=\cos t
$$

to a first-order system.

> Pause here. Let students commit to an answer before running the next cell.